# Phase 1 — Baseline MIP (No ML)

Establishes the pure-optimization reference point for all subsequent phases.
All logic lives in `mip/`; this notebook only calls it and displays results.

**What runs here:**
1. Load the synthetic data instance.
2. Compute the baseline (no-attack) cost matrix.
3. Solve the **deterministic** MIP (no scenarios).
4. Load attack scenarios, solve the **robust** MIP.
5. Visualise both solutions.

**Parameters to sweep (Phase 1 evaluation):** T, δ ∈ {0.05, 0.10}, K levels.

In [1]:
from pathlib import Path

import mip
import pandas as pd

from mip import (
    load_instance,
    compute_cost_matrix,
    load_scenario_batch,
    extract_deterministic_solution,
    extract_robust_solution,
)
from mip.models import build_deterministic_model, build_robust_model
from mip.viz import plot_solution, plot_service_levels, load_map_layers
from mip.scenarios import scenario_summary_df

# Project root derived from the installed mip package — no sys.path hacks needed.
PROJECT_ROOT = Path(mip.__file__).parent.parent

## 1. Load instance

In [2]:
instance = load_instance()

print(f"Coarse graph : {instance.CG.number_of_nodes()} nodes, {instance.CG.number_of_edges()} edges")
print(f"Demand nodes : {len(instance.D)}")
print(f"Total demand : {sum(instance.demand.values()):.2f} (daily avg)")
print(f"a_i range    : {min(instance.a.values()):.2f} – {max(instance.a.values()):.2f}")
print(f"b_i range    : {min(instance.b.values()):.2f} – {max(instance.b.values()):.2f}")

instance.demand_df.head()

Coarse graph : 112 nodes, 193 edges
Demand nodes : 14
Total demand : 162.05 (daily avg)
a_i range    : 4.51 – 99.56
b_i range    : 0.11 – 2.49


,coarse_node,demand_amount,total_fatalities,plot_lat,plot_lon
0,0,14.576712,45414,46.673281,32.586294
1,4,2.454795,791,46.625397,30.834090
2,8,10.420548,4129,47.068261,33.627809
3,18,6.168493,2236,51.824653,31.942545
4,21,7.761644,145,52.034644,33.850046


## 2. Compute baseline cost matrix

In [ ]:
c = compute_cost_matrix(instance)

import numpy as np
vals = list(c.values())
print(f"Cost matrix  : {len(c)} (i,j) pairs")
print(f"Travel times : min={min(vals):.3f}h  mean={np.mean(vals):.3f}h  max={max(vals):.3f}h")

## 3. Deterministic MIP (no scenarios)

Pure cost minimisation — no attack scenarios, no chance constraint.  
This is the cheapest possible design and the lower bound for build cost.

In [ ]:
T_det = 3.0  # weighted-average travel-time threshold (hours)

det_model, det_vars = build_deterministic_model(instance, c, T=T_det, verbose=True)
det_model.optimize()

det_result = extract_deterministic_solution(det_model, det_vars, instance, c, T=T_det)
print(f"Open hubs : {len(det_result.open_hubs)}")
print(f"Obj value : {det_result.obj_val:.2f}")
print(f"Avg tt    : {det_result.avg_travel_time:.3f} h")

In [ ]:
borders, occupied = load_map_layers()
fig = plot_solution(instance, det_result, borders, occupied, title="Deterministic solution (no scenarios)")
fig.savefig(PROJECT_ROOT / "outputs" / "phase1_deterministic.png", dpi=150, bbox_inches="tight")

## 4. Load attack scenarios

SCENARIO_ROOT = PROJECT_ROOT / "data" / "attack_scenarios" / "batch_2026-03-30_set1"

scenarios = load_scenario_batch(SCENARIO_ROOT, instance)

print(f"Loaded {len(scenarios)} scenarios")
scenario_summary_df(scenarios)

SCENARIO_ROOT = PROJECT_ROOT / "data" / "attack_scenarios" / "batch_2026-03-30_set1"

scenarios = load_scenario_batch(SCENARIO_ROOT, instance)

print(f"Loaded {len(scenarios)} scenarios")
scenario_summary_df(scenarios)

In [ ]:
DELTA = 0.05   # at most 5% of scenarios may violate threshold

rob_model, rob_vars = build_robust_model(
    instance, scenarios, delta=DELTA, M_service=100.0, verbose=True
)
rob_model.optimize()

In [ ]:
# delta=0.10 → violation budget = 10 * 0.10 = 1 scenario may be violated (meaningful chance constraint)
# delta=0.05 → budget = 0.5 → 0 violations (fully robust — no chance constraint slack)
DELTA = 0.10

rob_model, rob_vars = build_robust_model(
    instance, scenarios, delta=DELTA, M_service=100.0, verbose=True
)
rob_model.optimize()

rob_result = extract_robust_solution(rob_model, rob_vars, instance, scenarios, delta=DELTA)
print(f"Open hubs  : {len(rob_result.open_hubs)}")
print(f"Obj value  : {rob_result.obj_val:.2f}")
print(f"Violations : {len(rob_result.violated_scenarios)} / {len(scenarios)}")

In [ ]:
fig_map = plot_solution(instance, rob_result, borders, occupied, title=f"Robust solution — δ={DELTA}")
fig_map.savefig(PROJECT_ROOT / "outputs" / "phase1_robust_map.png", dpi=150, bbox_inches="tight")

fig_svc = plot_service_levels(rob_result)
fig_svc.savefig(PROJECT_ROOT / "outputs" / "phase1_robust_service.png", dpi=150, bbox_inches="tight")

In [ ]:
summary = pd.DataFrame([
    {
        "model": "Deterministic",
        "scenarios": 0,
        "delta": None,
        "open_hubs": len(det_result.open_hubs),
        "fixed_cost": det_result.fixed_cost,
        "capacity_cost": det_result.capacity_cost,
        "total_cost": det_result.obj_val,
        "avg_travel_time_hr": det_result.avg_travel_time,
    },
    {
        "model": "Robust",
        "scenarios": len(scenarios),
        "delta": DELTA,
        "open_hubs": len(rob_result.open_hubs),
        "fixed_cost": rob_result.fixed_cost,
        "capacity_cost": rob_result.capacity_cost,
        "total_cost": rob_result.obj_val,
        "avg_travel_time_hr": None,
    },
])

summary